# Ask questions from your own data (small transformer LLM)

Put `data.txt` in the same folder (in Colab, upload it with the folder icon), edit the `questions` list in cell 2, and run all cells.

What happens:
1. A BPE tokenizer is trained on `data.txt`.
2. Every sentence of `data.txt` becomes one training sequence: `[BOS]` + the sentence tokens + `[EOS]`.
3. Sentences have different lengths, so each batch is padded with `[PAD]` to the same length. A padding mask tells attention to ignore the padding, and the loss ignores it too.
4. A small GPT style transformer (token embedding + position embedding, then transformer blocks with causal self attention) is trained to predict the next token at every position.
5. For each question, the code finds the sentence in `data.txt` that shares the most words with the question.
6. The model then continues from the first few tokens of that sentence, so its answer comes from what it learned from your data.

Why step 5 exists: this is a tiny model, so it cannot understand a question by itself. Matching the question to a sentence gives it the right place to start from. This is a plain word matching step, not part of the model.

In [ ]:
# If a library is missing (local VS Code only), run this once:
# %pip install tokenizers torch

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
# ===== EDIT THIS PART =====
data_file = "data.txt"

questions = [
    "What is the Vadhavan port?",
    "How do I register on e-Shram?",
    "Which fruit is Dahanu famous for?"
]

vocab_size = 5000          # maximum tokens the tokenizer may learn
block_size = 64            # longest input the model can read (also the number of positions)
batch_size = 16            # sentences per training step
embedding_dim = 128        # size of every token vector (d_model)
num_heads = 4              # attention heads (embedding_dim must divide by this)
num_layers = 3             # number of transformer blocks
ff_size = 512              # width of the feed forward layer inside each block
dropout = 0.0              # keep 0 so the model can memorise the data
learning_rate = 0.001
max_iterations = 3000
seed_tokens = 8            # how many tokens of the matched sentence the model starts from
validation_fraction = 0.0  # 0 = train on every sentence (best for answering from your data)
random_seed = 42
# ==========================

In [ ]:
torch.manual_seed(random_seed)

# Load the data
with open(data_file, "r", encoding="utf-8") as file:
    text = file.read()

# Split the text into sentences (a full stop followed by a space)
sentences = []
for s in text.replace("\n", " ").split(". "):
    s = s.strip()
    if len(s.split()) > 3:
        sentences.append(s)

print("Corpus characters:", len(text))
print("Sentences:", len(sentences))

## Tokenization

Same BPE tokenizer as before, with two small additions: a `##` marker on word pieces and a matching decoder, so that decoded text joins word pieces back into whole words (for example `famous` instead of `fam ous`).

In [ ]:
# Train the tokenizer on data.txt
tokenizer = Tokenizer(BPE(unk_token="[UNK]", continuing_subword_prefix="##"))

tokenizer.pre_tokenizer = Whitespace()
tokenizer.decoder = decoders.WordPiece(prefix="##", cleanup=False)

trainer = BpeTrainer(
    vocab_size = vocab_size,
    min_frequency = 2,
    continuing_subword_prefix = "##",
    special_tokens = ["[UNK]", "[BOS]", "[EOS]", "[PAD]"]
)

tokenizer.train([data_file], trainer=trainer)

vocab_size = tokenizer.get_vocab_size()

# Ids of the special tokens
bos_id = tokenizer.token_to_id("[BOS]")   # beginning of a sentence
eos_id = tokenizer.token_to_id("[EOS]")   # end of a sentence
pad_id = tokenizer.token_to_id("[PAD]")   # filler so that every row in a batch has the same length

# One training sequence per sentence: [BOS] tokens... [EOS]
sequences = []
too_long = 0

for s in sentences:
    ids = [bos_id] + tokenizer.encode(s).ids + [eos_id]

    # The model reads at most block_size tokens, so a sequence can be block_size + 1 long
    if len(ids) > block_size + 1:
        ids = ids[:block_size + 1]
        too_long = too_long + 1

    sequences.append(ids)

lengths = [len(q) for q in sequences]

print("Vocabulary size:", vocab_size)
print("Sequences:", len(sequences))
print("Shortest / longest sequence:", min(lengths), "/", max(lengths))
print("Total real tokens:", sum(lengths))

if too_long > 0:
    print("Warning:", too_long, "sentences were cut because they are longer than block_size. Increase block_size.")

## Padding and attention masking

Batches must be rectangular, so shorter sentences are filled with `[PAD]`. `pad_mask` is `True` wherever there is padding. The model uses it so that real tokens never attend to padding, and the loss skips padded targets.

In [ ]:
# Split into training and validation sequences
order = torch.randperm(len(sequences)).tolist()
val_count = int(validation_fraction * len(sequences))

val_sequences = [sequences[i] for i in order[:val_count]]
train_sequences = [sequences[i] for i in order[val_count:]]

print("Training sequences:", len(train_sequences))
print("Validation sequences:", len(val_sequences))


# Batch maker: x = input tokens, y = the same tokens shifted by one (the next token at every position)
def get_batch(data, batch_size=16):
    picks = torch.randint(len(data), (batch_size,)).tolist()
    batch = [data[i] for i in picks]

    # Padded length = longest sequence in this batch minus 1 (inputs and targets are shifted by one)
    longest = max(len(q) for q in batch) - 1

    x = torch.full((batch_size, longest), pad_id, dtype=torch.long)
    y = torch.full((batch_size, longest), pad_id, dtype=torch.long)

    for row, q in enumerate(batch):
        n = len(q) - 1
        x[row, :n] = torch.tensor(q[:-1])   # input: everything except the last token
        y[row, :n] = torch.tensor(q[1:])    # target: everything except the first token

    # True where x is padding
    pad_mask = (x == pad_id)

    return x, y, pad_mask


# Look at one small batch to see the padding
x, y, pad_mask = get_batch(train_sequences, batch_size=3)

print("x shape:", tuple(x.shape))
for row in range(3):
    print("-" * 60)
    print("x tokens :", tokenizer.decode(x[row].tolist(), skip_special_tokens=False))
    print("x ids    :", x[row].tolist())
    print("pad_mask :", pad_mask[row].int().tolist())

## Transformer model

- **Token embedding**: one vector per token id (what the token is).
- **Position embedding**: one vector per position 0 to `block_size - 1` (where the token is). A transformer reads all tokens at once, so without this it would not know the order.
- **Causal mask**: position `i` may only look at positions `0..i`. Without it the model could peek at the next token it is supposed to predict.
- **Padding mask**: padded positions are ignored by attention.
- **Transformer blocks**: each block has self attention (every token looks at the earlier tokens and weighs how relevant each is) and a feed forward network, with residual connections and layer normalisation.
- **Output head**: turns each position's vector into one score per vocabulary token.

In [ ]:
class SimpleTransformerLM(nn.Module):

    def __init__(self, vocab_size, block_size, embedding_dim, num_heads, num_layers, ff_size, dropout):
        super().__init__()

        # Token embedding: what the token is
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)

        # Position embedding: where the token is
        self.position_embedding = nn.Embedding(block_size, embedding_dim)

        self.drop = nn.Dropout(dropout)

        # One transformer block: self attention + feed forward (norm_first = layer norm before each part)
        block = nn.TransformerEncoderLayer(
            d_model = embedding_dim,
            nhead = num_heads,
            dim_feedforward = ff_size,
            dropout = dropout,
            activation = "gelu",
            batch_first = True,
            norm_first = True
        )

        # Stack several blocks
        self.blocks = nn.TransformerEncoder(block, num_layers=num_layers, enable_nested_tensor=False)

        # Final layer normalisation and output layer
        self.final_norm = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x, pad_mask=None):

        batch, length = x.shape

        # Token embeddings: (batch, length, embedding_dim)
        token_vectors = self.token_embedding(x)

        # Position embeddings for positions 0, 1, 2 ... length-1
        positions = torch.arange(length, device=x.device)
        position_vectors = self.position_embedding(positions)

        # Every token vector now carries its meaning and its position
        h = self.drop(token_vectors + position_vectors)

        # Causal mask: True above the diagonal means "not allowed to look at the future"
        causal_mask = torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=x.device),
            diagonal=1
        )

        # Transformer blocks (padding is ignored through pad_mask)
        h = self.blocks(h, mask=causal_mask, src_key_padding_mask=pad_mask)

        h = self.final_norm(h)

        # Next token scores at every position: (batch, length, vocab_size)
        logits = self.head(h)

        return logits


model = SimpleTransformerLM(vocab_size, block_size, embedding_dim, num_heads, num_layers, ff_size, dropout)

parameter_count = sum(p.numel() for p in model.parameters())
print("Parameters:", parameter_count)
print(model)

## Training

At every position the model predicts the next token. `ignore_index=pad_id` makes the loss skip padded targets, so padding teaches the model nothing.

In [ ]:
# Loss that skips padding
def compute_loss(logits, y):
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1),
        ignore_index = pad_id
    )


# Average loss over several random batches
def evaluate(model, data, batches=20):

    if len(data) == 0:
        return None

    model.eval()
    total_loss = 0

    with torch.no_grad():

        for _ in range(batches):

            x, y, pad_mask = get_batch(data, batch_size)

            logits = model(x, pad_mask)

            total_loss += compute_loss(logits, y).item()

    model.train()

    return total_loss / batches


# Train the model
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iteration in range(max_iterations):

    x, y, pad_mask = get_batch(train_sequences, batch_size)

    optimizer.zero_grad()

    logits = model(x, pad_mask)

    loss = compute_loss(logits, y)

    loss.backward()

    optimizer.step()

    if iteration % 250 == 0:
        print(f"Iteration: {iteration}, Loss: {loss.item():.4f}")

print("Training Loss:", evaluate(model, train_sequences))

if len(val_sequences) > 0:
    print("Validation Loss:", evaluate(model, val_sequences))

In [ ]:
# Find the sentence in data.txt that best matches the question
stop_words = ["what", "is", "the", "of", "in", "a", "an", "and", "to", "for", "are",
              "how", "which", "who", "where", "when", "does", "do", "can", "i", "me",
              "on", "at", "by", "with", "from", "about", "it", "this", "that", "as",
              "be", "or", "tell", "was", "were", "has", "have", "my", "your", "you"]


def clean_words(text):

    text = text.lower()

    for ch in [".", ",", "?", "!", ":", ";", "(", ")", '"', "'", "-", "/"]:
        text = text.replace(ch, " ")

    result = []

    for w in text.split():
        if w not in stop_words:
            result.append(w)

    return result


def find_best_sentence(question):

    question_words = clean_words(question)

    best_sentence = ""
    best_score = 0

    for s in sentences:

        sentence_words = clean_words(s)

        score = 0

        for w in question_words:
            if w in sentence_words:
                score = score + 1

        if score > best_score:
            best_score = score
            best_sentence = s

    return best_sentence, best_score

In [ ]:
# Let the model continue from the start of a sentence (one token at a time)
def generate_from_ids(ids, max_new_tokens):

    model.eval()

    with torch.no_grad():

        for _ in range(max_new_tokens):

            # The model can read at most block_size tokens: keep the most recent ones
            input_ids = torch.tensor(ids[-block_size:], dtype=torch.long).unsqueeze(0)

            # One sentence, so there is no padding here and no pad mask is needed
            logits = model(input_ids)

            # Only the last position predicts the next token
            predicted_id = torch.argmax(logits[0, -1]).item()

            # [EOS] means the sentence is finished
            if predicted_id == eos_id:
                break

            ids.append(predicted_id)

    model.train()

    return ids


def answer_question(question):

    best_sentence, score = find_best_sentence(question)

    print("Question:", question)

    if score == 0:
        print("Answer: I could not find this in the data.")
        print("-" * 60)
        return

    print("Best match in data:", best_sentence)

    ids = tokenizer.encode(best_sentence).ids

    # Start from [BOS] + the first seed_tokens tokens of the matched sentence
    start_ids = [bos_id] + ids[:seed_tokens]

    new_ids = generate_from_ids(start_ids, max_new_tokens=block_size)

    print("Model's answer:", tokenizer.decode(new_ids, skip_special_tokens=True))
    print("-" * 60)

In [ ]:
# Answer the questions from the list
for question in questions:
    answer_question(question)

In [ ]:
# Ask your own question (type "exit" to stop)
while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        break

    answer_question(question)